# 11 — Full Fine-Tune

The payoff: train the final ARO model on the complete assembled dataset.

This is a full LoRA fine-tune using the same rank (16) and depth (16 layers) as the
warm-start adapter from notebook 04 — so the two adapters are architecturally compatible
and the final model builds consistently on the warm-start's foundation.

The fused output is a standalone model that needs no adapter at inference time.
It becomes the base for iterative self-improvement rounds in notebook 13.

**Why not just use the warm-start adapter?** The warm-start was trained on ~500–1000
curated pairs. The full dataset assembled by notebook 10 typically contains 2000–5000
samples across seven task types, including execution-grounded pairs, error patterns,
and wiki Q&A that the warm-start never saw. The full fine-tune integrates all of it.

**Install:** `pip install mlx-lm`
**Input:**  `../data/05_dataset/mlx/` (train.jsonl + valid.jsonl, from notebook 10)
**Output:** `../models/round_0/adapter/` (LoRA adapter)
            `../models/round_0/fused/`   (merged weights — base for notebook 13)

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'mlx-lm'], check=False)

In [ ]:
import json
import sys
from pathlib import Path

try:
    SCRIPT_DIR = Path(__file__).parent.resolve()
except NameError:
    SCRIPT_DIR = Path('.').resolve()

DATA_DIR    = SCRIPT_DIR / '../data/05_dataset/mlx'
MODELS_DIR  = SCRIPT_DIR / '../models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# --- Config ---
BASE_MODEL    = 'mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit'
ROUND         = 0          # increment for each iterative round (see notebook 08)
ADAPTER_DIR   = MODELS_DIR / f'round_{ROUND}' / 'adapter'
FUSED_DIR     = MODELS_DIR / f'round_{ROUND}' / 'fused'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

with open(SCRIPT_DIR / '../data/05_dataset/stats.json') as f:
    stats = json.load(f)

print(f'Base model:   {BASE_MODEL}')
print(f'Train/valid:  {stats["train"]} / {stats["valid"]} samples')
print(f'Adapter out:  {ADAPTER_DIR}')

## Training config

In [ ]:
# Tune these to your Mac's RAM:
#   16 GB: batch_size=4, lora_layers=16, lora_rank=16  (safe)
#   32 GB: batch_size=8, lora_layers=16, lora_rank=16  (fast)
#   8 GB:  batch_size=2, lora_layers=8,  lora_rank=8   (tight but works with 4-bit model)
#
# lora_layers=16 and lora_rank=16 match the warm-start adapter settings from
# notebook 04, so the final fine-tune builds consistently on top of it.

BATCH_SIZE    = 4
GRAD_ACCUM    = 4         # effective batch = BATCH_SIZE × GRAD_ACCUM = 16
LORA_LAYERS   = 16        # transformer layers to apply LoRA (matches NB04 warm-start)
LORA_RANK     = 16        # LoRA rank — higher = more capacity (matches NB04 warm-start)
LEARNING_RATE = 2e-4
ITERS         = 600       # ~3 epochs for a 1000-sample dataset; increase for more data
SAVE_EVERY    = 200       # save checkpoint every N iters
VAL_BATCHES   = 20        # how many validation batches to run

print(f'Effective batch size: {BATCH_SIZE * GRAD_ACCUM}')
print(f'LoRA layers/rank:     {LORA_LAYERS} / {LORA_RANK}')
print(f'Iterations:           {ITERS}')

## Run LoRA training

In [ ]:
train_cmd = [
    sys.executable, '-m', 'mlx_lm', 'lora',
    '--model',                   BASE_MODEL,
    '--data',                    str(DATA_DIR),
    '--train',
    '--batch-size',              str(BATCH_SIZE),
    '--grad-accumulation-steps', str(GRAD_ACCUM),
    '--num-layers',              str(LORA_LAYERS),
    '--lora-rank',               str(LORA_RANK),
    '--learning-rate',           str(LEARNING_RATE),
    '--iters',                   str(ITERS),
    '--save-every',              str(SAVE_EVERY),
    '--val-batches',             str(VAL_BATCHES),
    '--adapter-path',            str(ADAPTER_DIR),
    '--mask-prompt',             # compute loss only on assistant turns
]

print('Starting training...')
print(' '.join(train_cmd))
print()

result = subprocess.run(train_cmd)  # streams output live

if result.returncode != 0:
    raise RuntimeError(f'Training failed with exit code {result.returncode}')

print(f'\nTraining complete. Adapter saved to: {ADAPTER_DIR}')

## Fuse adapter into base weights

In [ ]:
# Fusing creates a standalone model (no adapter needed at inference time).
# This fused model is used as the base for the next iterative round.

fuse_cmd = [
    sys.executable, '-m', 'mlx_lm.fuse',
    '--model',        BASE_MODEL,
    '--adapter-path', str(ADAPTER_DIR),
    '--save-path',    str(FUSED_DIR),
]

print('Fusing adapter into base weights...')
print(' '.join(fuse_cmd))
result = subprocess.run(fuse_cmd)

if result.returncode != 0:
    raise RuntimeError(f'Fuse failed with exit code {result.returncode}')

print(f'Fused model saved to: {FUSED_DIR}')

## Smoke test: generate one sample with fine-tuned model

In [ ]:
from mlx_lm import load, generate as mlx_generate

print(f'Loading fused model from {FUSED_DIR} ...')
model, tokenizer = load(str(FUSED_DIR))

test_messages = [
    {'role': 'system',  'content': 'You are an expert ARO language programmer.'},
    {'role': 'user',    'content': 'Write an ARO feature set that retrieves a user by id and returns an OK response.'},
]
prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
response = mlx_generate(model, tokenizer, prompt=prompt, max_tokens=300, verbose=False)

print('\n=== Smoke test output ===')
print(response)

## Save round metadata

In [ ]:
meta = {
    'round':         ROUND,
    'base_model':    BASE_MODEL,
    'adapter_dir':   str(ADAPTER_DIR),
    'fused_dir':     str(FUSED_DIR),
    'train_samples': stats['train'],
    'iters':         ITERS,
    'batch_size':    BATCH_SIZE,
    'lora_layers':   LORA_LAYERS,
    'lora_rank':     LORA_RANK,
    'learning_rate': LEARNING_RATE,
}
with open(MODELS_DIR / f'round_{ROUND}' / 'meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'\nRound {ROUND} complete.')
print(f'  Next step: run notebook 13 for iterative self-improvement rounds,')
print(f'             or notebook 12 for evaluation.')